In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets

from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

print("Loading base Llama 3 (8B) quantized model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Adding LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Model loaded successfully")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kp71wlu5/unsloth_d2e5866cccb54ac0b7b21a605d300daa
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kp71wlu5/unsloth_d2e5866cccb54ac0b7b21a605d300daa
  Resolved https://github.com/unslothai/unsloth.git to commit db3393fbc857af6e8fbcab643a95d212b81c8010
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.0 MB/s eta 0:00:0

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


2. Adding LoRA adapters (The 'medical brain')...


Unsloth 2026.5.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model ready to learn medicine!


In [ ]:
from google.colab import drive
from datasets import load_from_disk

print("Mounting Google Drive...")
drive.mount('/content/drive')

# NOTE: Verify this path matches yours exactly
ruta_dataset = '/content/drive/MyDrive/Bioinformática_LLM/data/processed_data/medical_llama3_instruct_dataset_short'

print(f"Loading dataset from: {ruta_dataset}")
dataset = load_from_disk(ruta_dataset)
print("Dataset loaded successfully. Sample:")
print(dataset)

Mounting Google Drive...
Mounted at /content/drive
Loading dataset from: /content/drive/MyDrive/Bioinformática_LLM/data/processed_data/medical_llama3_instruct_dataset_short
Dataset loaded successfully! Here's a sample:
DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction', 'prompt'],
        num_rows: 2000
    })
})


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Configuring Fine-Tuning engine with updated libraries...")

# Unsloth requires a formatting function
def formatting_func(examples):
    # Si la librería envía una lista de prompts
    if isinstance(examples["prompt"], list):
        return examples["prompt"]
    # string individual en caso de que unsloth use sola una fila en su chequeo inicial
    return [examples["prompt"]]

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    formatting_func = formatting_func,  # REQUIRED by Unsloth
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 120,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("\nSTARTING TRAINING")
trainer_stats = trainer.train()
print("\n" + "="*40)
print("TRAINING COMPLETED SUCCESSFULLY")
print("="*40)

Configuring Fine-Tuning engine with updated libraries...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]


STARTING REAL TRAINING! 🚀
You'll see a progress bar. The 'Training Loss' column will start decreasing.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.644550
10,1.736851
15,1.668147
20,1.435959
25,1.568303
30,1.591609
35,1.268565
40,1.211057
45,1.312389
50,1.528956


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-120/tokenizer_config.json.



TRAINING COMPLETED SUCCESSFULLY!


In [ ]:
print("\nVerification:")
print(f"Model type: {type(model)}")
print(f"Tokenizer type: {type(tokenizer)}")
print(f"Tokenizer is not None: {tokenizer is not None}")
print(f"Dataset: {dataset}")
print(f"Train split: {dataset['train']}")
print(f"Train split type: {type(dataset['train'])}")
print(f"Columns: {dataset['train'].column_names}")
print(f"First prompt sample:\n{dataset['train'][0]['prompt'][:500]}...")

# ================================================================
# 5. Evaluación
# ================================================================
import pandas as pd
import time
from copy import deepcopy
import os

FastLanguageModel.for_inference(model)

# Los mismos 10 pacientes que en el modelo normal
patients_base = [
    {
        "id_base": "Case_001_Lupus",
        "specialty": "Rheumatology",
        "real_disease": "Systemic Lupus Erythematosus (SLE)",
        "symptoms": "28-year-old woman. Presents with migratory joint pain, extreme fatigue of months of evolution, painless oral ulcers, and a reddish rash on the cheeks that worsens with sun exposure."
    },
    {
        "id_base": "Case_002_Pheochromocytoma",
        "specialty": "Endocrinology",
        "real_disease": "Pheochromocytoma",
        "symptoms": "45-year-old man. Presents with sudden episodes of throbbing headache, profuse sweating, and severe palpitations lasting 15 minutes. During the episode in the emergency room, his blood pressure is 190/110 mmHg."
    },
    {
        "id_base": "Case_003_Endocarditis",
        "specialty": "Infectious Diseases/Cardiology",
        "real_disease": "Infective Endocarditis",
        "symptoms": "35-year-old man, intravenous drug user. Presents with intermittent fever for 3 weeks, appearance of a new heart murmur, and small reddish lesions on the palms (Janeway lesions)."
    },
    {
        "id_base": "Case_004_Myasthenia",
        "specialty": "Neurology",
        "real_disease": "Myasthenia Gravis",
        "symptoms": "32-year-old woman. Reports double vision (diplopia) and drooping eyelids (ptosis) that worsen notably at the end of the day or after reading for a while. Recently has difficulty swallowing solids."
    },
    {
        "id_base": "Case_005_Embolism",
        "specialty": "Pulmonology",
        "real_disease": "Pulmonary Thromboembolism (PTE)",
        "symptoms": "60-year-old woman, hip surgery 10 days ago. Presents with sudden onset dyspnea, chest pain that worsens with deep inspiration (pleuritic), and tachycardia at 120 bpm. No fever."
    },
    {
        "id_base": "Case_006_Lyme",
        "specialty": "Infectious Diseases",
        "real_disease": "Lyme Disease",
        "symptoms": "40-year-old man, frequent camper. Presents with fever, myalgias, and an expanding target-shaped skin lesion (erythema migrans) on the thigh, without significant pain or itching."
    },
    {
        "id_base": "Case_007_Sclerosis",
        "specialty": "Neurology",
        "real_disease": "Multiple Sclerosis",
        "symptoms": "25-year-old woman. Painful vision loss in the left eye (optic neuritis) established in 2 days. Reports that a year ago she had an episode of tingling in both legs that disappeared on its own."
    },
    {
        "id_base": "Case_008_Addison",
        "specialty": "Endocrinology",
        "real_disease": "Addison's Disease",
        "symptoms": "50-year-old man. Presents with extreme fatigue, weight loss, unusual craving for salty foods, and hyperpigmentation (darkening) in the hand folds and gums. Low blood pressure (85/50)."
    },
    {
        "id_base": "Case_009_Ectopic",
        "specialty": "Gynecology",
        "real_disease": "Ruptured Ectopic Pregnancy",
        "symptoms": "29-year-old woman. Missed period for 6 weeks. Presents with acute, sharp abdominal pain in the right iliac fossa, intense dizziness, and pain radiating to the right shoulder."
    },
    {
        "id_base": "Case_010_GuillainBarre",
        "specialty": "Neurology",
        "real_disease": "Guillain-Barré Syndrome",
        "symptoms": "38-year-old man. Two weeks ago had gastroenteritis. Now presents with progressive muscle weakness that started in the feet and is ascending toward the thighs, with absence of patellar reflexes."
    }
]

# 10 intentos por pacientes para un total de 100 intentos
patients_test = []
for attempt in range(1, 11):
    for patient in patients_base:
        clone = deepcopy(patient)
        clone["id"] = f"{clone['id_base']}_Attempt_{attempt:02d}"
        patients_test.append(clone)

patients_test = sorted(patients_test, key=lambda x: x['id'])
evaluation_results = []

print(f"Starting evaluation with FINE-TUNED model: {len(patients_test)} inferences...\n")


for i, patient in enumerate(patients_test):
    print(f"[{i+1}/100] Processing {patient['id']}...")

    prompt_dynamic = f"""You are an expert physician. Analyze this complex clinical case rigorously.

Clinical Case: {patient['symptoms']}

Provide exclusively:
1. Main diagnostic suspicion.
2. Brief medical justification.

Response:"""

    inputs = tokenizer([prompt_dynamic], return_tensors="pt").to("cuda")
    start_time = time.time()

    # TEMPERATURA 0.7
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    elapsed_time = time.time() - start_time
    full_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    clean_response = full_response[len(prompt_dynamic):].strip()

    evaluation_results.append({
        "ID_Case": patient["id"],
        "Specialty": patient["specialty"],
        "Real_Diagnosis": patient["real_disease"],
        "Response_FINETUNED_Model": clean_response,
        "Inference_Time_sec": round(elapsed_time, 2)
    })

# Guardado resultados en Drive
output_dir = '/content/drive/MyDrive/Bioinformática_LLM'
os.makedirs(output_dir, exist_ok=True)

save_path = f'{output_dir}/resultados_modelo_finetuned_complejos.csv'
df_results = pd.DataFrame(evaluation_results)
df_results.to_csv(save_path, index=False, encoding='utf-8')

print("\n" + "="*50)
print(f"Saved to: {save_path}")
print(f"\nFirst 5 results preview:")
display(df_results.head())

# ================================================================
# 6. Guardar modelo fine tunned
# ================================================================
print("\nSaving fine-tuned model...")
model_save_path = '/content/drive/MyDrive/Bioinformática_LLM/modelo_finetuned_lora'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to: {model_save_path}")

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Verification:
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Tokenizer type: <class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
Tokenizer is not None: True
Dataset: DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction', 'prompt'],
        num_rows: 2000
    })
})
Train split: Dataset({
    features: ['output', 'input', 'instruction', 'prompt'],
    num_rows: 2000
})
Train split type: <class 'datasets.arrow_dataset.Dataset'>
Columns: ['output', 'input', 'instruction', 'prompt']
First prompt sample:
<|start_header_id|>system<|end_header_id|> Answer the question truthfully, you are a medical professional.<|eot_id|><|start_header_id|>user<|end_header_id|> This is the question: Can you provide an overview of the lung's squamous cell carcinoma?<|eot_id|><|start_header_id|>assistant<|end_header_id|> Squamous cell carcinoma of the lung may be classified according to the WHO histological classification system into 4 main types: p

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

[2/100] Processing Case_001_Lupus_Attempt_02...


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[100/100] Processing Case_010_GuillainBarre_Attempt_10...

Evaluation completed! Saved to: /content/drive/MyDrive/Bioinformática_LLM/resultados_modelo_finetuned_complejos.csv

First 5 results preview:


,ID_Case,Specialty,Real_Diagnosis,Response_FINETUNED_Model,Inference_Time_sec
0,Case_001_Lupus_Attempt_01,Rheumatology,Lupus Eritematoso Sistémico (LES),1. Systemic Lupus Erythematosus.\n2. Systemic ...,12.17
1,Case_001_Lupus_Attempt_02,Rheumatology,Lupus Eritematoso Sistémico (LES),1. Sjogren's syndrome\n2. Sjogren's syndrome i...,11.13
2,Case_001_Lupus_Attempt_03,Rheumatology,Lupus Eritematoso Sistémico (LES),1. Main diagnostic suspicion: Juvenile idiopat...,10.72
3,Case_001_Lupus_Attempt_04,Rheumatology,Lupus Eritematoso Sistémico (LES),1. Main diagnostic suspicion: Dermatitis polym...,10.62
4,Case_001_Lupus_Attempt_05,Rheumatology,Lupus Eritematoso Sistémico (LES),1. Suspected diagnosis: SLE (Systemic Lupus Er...,11.04



Saving fine-tuned model...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Bioinformática_LLM/modelo_finetuned_lora/tokenizer_config.json.


Model saved to: /content/drive/MyDrive/Bioinformática_LLM/modelo_finetuned_lora
